# 8.5. Batch Normalization

In the [last](https://github.com/DonaldKellett/my-ascend-notebooks/blob/90a2255c66346c38f16d7fc2e8d763a01592b38e/atomgit-ai/00-d2l-mindspore-ch8-modern-convolutional-neural-networks/03-multi-branch-networks-googlenet.ipynb) and previous chapters, we used [`mindspore.nn.BatchNorm2d`](https://www.mindspore.cn/docs/en/r2.9.0/api_python/nn/mindspore.nn.BatchNorm2d.html#mindspore.nn.BatchNorm2d) layers to stabilize the training of modern CNNs such as GoogLeNet without a solid understanding of what batch normalization is and how it actually works.

In this chapter, we'll take a closer look at the mathematics between batch normalization and understand why its discovery enabled modern deep learning practitioners to reliably train deep networks in excess of 100 layers without encountering vanishing or exploding gradients.

Unlike previous chapters involving modern deep networks which required the use of datacenter-level Ascend 910B4 training-optimized NPUs via the [AtomGit AI Notebook Lab](https://ai.gitcode.com/docs/notebooks/free-usage/) cloud environment to achieve reasonable training times, we'll simply implement our own batch normalization layer from first principles in this chapter and apply it to LeNet from chapter 7, see how it improves the validation loss and accuracy of our model. The entire training process should complete on the [OrangePi AIpro \(20T\)](http://www.orangepi.org/html/hardWare/computerAndMicrocontrollers/details/Orange-Pi-AIpro%2820t%29.html) featuring a single Ascend 310B1 NPU chip and core in around 30 minutes.

The software versions used in this notebook are listed below.

1. Ubuntu 22.04 LTS
1. Python 3.12
1. MindSpore 2.9.0
1. CANN 9.0.0

In [1]:
!npu-smi info

+--------------------------------------------------------------------------------------------------------+
| npu-smi 23.0.0                                   Version: 23.0.0                                       |
+-------------------------------+-----------------+------------------------------------------------------+
| NPU     Name                  | Health          | Power(W)     Temp(C)           Hugepages-Usage(page) |
| Chip    Device                | Bus-Id          | AICore(%)    Memory-Usage(MB)                        |
+===============================+=================+======================================================+
| 0       310B1                 | Alarm           | 0.0          51                1050  / 1050          |
| 0       0                     | NA              | 0            6885 / 23673                            |
+===============================+=================+======================================================+


In [2]:
!cat requirements.txt

absl-py==2.4.0
attrs==26.1.0
cloudpickle==3.1.2
decorator==5.2.1
jupyterlab==4.5.7
jupyterlab-git==0.53.0
jupyter-resource-usage==1.2.1
loguru==0.7.3
matplotlib==3.10.9
mindspore==2.9.0
ml-dtypes==0.5.4
msguard==0.0.8
openpyxl==3.1.5
opentelemetry-exporter-otlp-proto-grpc==1.33.1
opentelemetry-exporter-otlp-proto-http==1.33.1
pandas~=2.2
plotly>=5.11.0
pydantic==2.13.4
sympy==1.14.0
tornado==6.5.5


In [3]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.9.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 8.5.3. Implementation From Scratch

The key idea behind batch normalization is transforming each minibatch from its original mean $\mu$ and standard deviation $\sigma$ to a new mean $\beta$ and standard deviation $\gamma$. During training time, we compute the mean and standard deviation of each minibatch. However, during inference, we use the mean and standard deviation of the entire dataset instead as computed during training via a _moving_ average and standard deviation. In other words, the behavior of batch normalization differs between training and inference, similar to how dropout behaves as introduced in previous chapters.

A detailed exposition of the mathematics behind batch normalization is provided in [chapter 8.5](https://d2l.ai/chapter_convolutional-modern/batch-norm.html) of D2L. Here, we jump straight into the implementation.

In [5]:
import mindspore.ops as ops

def batch_norm(X, gamma, beta, moving_mean, moving_var, eps, momentum, training=False):
    if not training:
        # Inference mode - re-shape the distribution based on the moving mean and variance
        X_hat = (X - moving_mean) / ops.sqrt(moving_var + eps) # `eps` prevents division by zero issues
    else:
        # Training mode - re-shape the distribution based on the minibatch mean and standard deviation
        assert len(X.shape) in (2, 4) # Either (n, features) or NCHW
        if len(X.shape) == 2:
            # (n, features) - FC layer, calculate the mean and variance along the feature dimension
            mean = ops.mean(X, axis=0)
            var = ops.mean((X - mean) ** 2, axis=0)
        else:
            # NCHW - convolutional layer, calculate the mean and variance per channel across all samples and pixels
            mean = ops.mean(X, axis=(0, 2, 3), keep_dims=True) # Preserve dimensions for broadcasting
            var = ops.mean((X - mean) ** 2, axis=(0, 2, 3), keep_dims=True)
        X_hat = (X - mean) / ops.sqrt(var + eps)
        # Update the mean and variance via moving average
        moving_mean = (1.0 - momentum) * moving_mean + momentum * mean
        moving_var = (1.0 - momentum) * moving_var + momentum * var
    Y = gamma * X_hat + beta # Scale and shift
    return Y, ops.stop_gradient(moving_mean), ops.stop_gradient(moving_var)

Let's implement our `MyBatchNorm` layer. It handles both FC layers and convolutional layers based on the `batch_norm` function we defined above.

In general, it's often a good idea to extract the key mathematical calculations to a dedicated function like `batch_norm` and simply wrap it with a subclass of [`mindspore.nn.Cell`](https://www.mindspore.cn/docs/en/r2.9.0/api_python/nn/mindspore.nn.Cell.html) which handles the framework-level plumbing.

In [6]:
import mindspore.nn as nn

class MyBatchNorm(nn.Cell):
    def __init__(self, num_features, num_dims):
        super().__init__()
        if num_dims == 2:
            shape = (1, num_features)
        else:
            shape = (1, num_features, 1, 1)
        # Initialize the scaled mean and stddev to 0 and 1 respectively
        # These are learned parameters
        self.gamma = mindspore.Parameter(ops.ones(shape))
        self.beta = mindspore.Parameter(ops.zeros(shape))
        # Initialize the moving mean and variance to 0 and 1 respectively
        # Unlike `gamma` and `beta`, these are not learnable parameters and are not affected by backpropagation
        self.moving_mean = ops.zeros(shape)
        self.moving_var = ops.ones(shape)

    def construct(self, X):
        # Save the updated moving_mean and moving_var
        Y, self.moving_mean, self.moving_var = batch_norm(X,
                                                          self.gamma,
                                                          self.beta,
                                                          self.moving_mean,
                                                          self.moving_var,
                                                          eps=1e-5,
                                                          momentum=0.1,
                                                          training=self.training)
        return Y

## 8.5.4. LeNet with Batch Normalization

Recall LeNet from [chapter 7.6](https://d2l.ai/chapter_convolutional-neural-networks/lenet.html) of D2L. Let's insert our implementation of batch normalization after each convolutional layer + activation and see how it helps stabilize training and improves the validation loss and accuracy of LeNet.

In [7]:
lenet_my_bn = nn.SequentialCell([
    nn.Conv2d(1, 6, kernel_size=5),
    nn.Sigmoid(),
    MyBatchNorm(6, num_dims=4), # Apply BN after the activation
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5, pad_mode='valid'),
    nn.Sigmoid(),
    MyBatchNorm(16, num_dims=4),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Dense(400, 120, activation='sigmoid'),
    MyBatchNorm(120, num_dims=2), # BN for FC layers via `num_dims=2`
    nn.Dense(120, 84, activation='sigmoid'),
    MyBatchNorm(84, num_dims=2),
    nn.Dense(84, 10)
])
lenet_my_bn

SequentialCell(
  (0): Conv2d(input_channels=1, output_channels=6, kernel_size=(5, 5), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xe7ffb0f2d730>, bias_init=None, format=NCHW)
  (1): Sigmoid()
  (2): MyBatchNorm()
  (3): AvgPool2d(kernel_size=2, stride=2, pad_mode=VALID)
  (4): Conv2d(input_channels=6, output_channels=16, kernel_size=(5, 5), stride=(1, 1), pad_mode=valid, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xe7ff2e0fa030>, bias_init=None, format=NCHW)
  (5): Sigmoid()
  (6): MyBatchNorm()
  (7): AvgPool2d(kernel_size=2, stride=2, pad_mode=VALID)
  (8): Flatten()
  (9): Dense(
    input_channels=400, output_channels=120, has_bias=True, activation=Sigmoid()
    (activation): Sigmoid()
  )
  (10): MyBatchNorm()
  (11): Dense(
    input_channels=120, output_channels=84, has_bias=True, activation=Sigmoid()
    (

Now for the training process.

In [8]:
import os

dataset_dir = 'data/fashion/'
os.makedirs(dataset_dir, exist_ok=True)

In [9]:
import gzip
import urllib.request

prefix_url = 'https://donaldsebleung.com/assets/datasets/fashion-mnist'
X_train_url = f'{prefix_url}/train-images-idx3-ubyte.gz'
y_train_url = f'{prefix_url}/train-labels-idx1-ubyte.gz'
X_test_url = f'{prefix_url}/t10k-images-idx3-ubyte.gz'
y_test_url = f'{prefix_url}/t10k-labels-idx1-ubyte.gz'

X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

with urllib.request.urlopen(X_train_url) as response:
    with open(X_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_train_url) as response:
    with open(y_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(X_test_url) as response:
    with open(X_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_test_url) as response:
    with open(y_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

In [10]:
import mindspore.dataset as ds

train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [11]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import dtype as mstype

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(28, 28)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=128, drop_remainder=False)
    return dataset

train_ds = transform_ds(dataset=train_ds)
test_ds = transform_ds(dataset=test_ds)

In [12]:
import mindspore.amp as amp

lenet_my_bn_amp = amp.auto_mixed_precision(network=lenet_my_bn, amp_level='O2')
lenet_my_bn_amp

_OutputTo32(
  (_backbone): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=6, kernel_size=(5, 5), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xe7ffb0f2d730>, bias_init=None, format=NCHW)
    (1): Sigmoid()
    (2): MyBatchNorm()
    (3): AvgPool2d(kernel_size=2, stride=2, pad_mode=VALID)
    (4): Conv2d(input_channels=6, output_channels=16, kernel_size=(5, 5), stride=(1, 1), pad_mode=valid, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xe7ff2e0fa030>, bias_init=None, format=NCHW)
    (5): Sigmoid()
    (6): MyBatchNorm()
    (7): AvgPool2d(kernel_size=2, stride=2, pad_mode=VALID)
    (8): Flatten()
    (9): Dense(
      input_channels=400, output_channels=120, has_bias=True, activation=Sigmoid()
      (activation): Sigmoid()
    )
    (10): MyBatchNorm()
    (11): Dense(
      input_channels=120, o

In [13]:
loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
loss_fn

SoftmaxCrossEntropyWithLogits()

In [14]:
optimizer = nn.SGD(params=lenet_my_bn_amp.trainable_params(), learning_rate=0.1)
optimizer

SGD()

In [15]:
from mindspore.train import Model

model = Model(network=lenet_my_bn_amp,
              loss_fn=loss_fn,
              optimizer=optimizer,
              metrics={'accuracy', 'loss'})
model

In [16]:
from mindspore.train import EarlyStopping

early_stopping = EarlyStopping(patience=5, verbose=True, restore_best_weights=True)
early_stopping

In [17]:
epochs = 100

In [18]:
%%time
model.fit(epoch=epochs,
          train_dataset=train_ds,
          valid_dataset=test_ds,
          callbacks=[early_stopping])

/usr/local/Ascend/cann-9.0.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:179: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/c

.path string is NULLpath string is NULL.

/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:97: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)
/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:157: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)
/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:97: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)
/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:157: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)


.

/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:97: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)
/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:157: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)


...

/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:97: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)
/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:157: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)
/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:97: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)
/usr/local/Ascend/cann-9.0.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:157: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)


..Restoring model weights from the end of the best epoch.
Epoch 00010: early stopping
CPU times: user 26min 4s, sys: 3min 12s, total: 29min 16s
Wall time: 14min 15s


Let's check the validation loss and accuracy of our improved LeNet with batch normalization.

In [19]:
val_metrics = model.eval(valid_dataset=test_ds)
val_loss = val_metrics['loss']
val_accuracy = val_metrics['accuracy']
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_accuracy:.4f}')

Validation loss: 0.5155
Validation accuracy: 0.8079


With batch normalization, the model converges with fewer epochs and the accuracy is increased from just over $75\%$ to around $80\%$. Excellent!

## 8.5.5. Concise Implementation

TODO